<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/60_linear_algebra_2/230_QR_Method_Eigenvalues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)



# QR 알고리듬으로 고유치 구하기<br>Finding Eigenvalues by the QR Algorithm


이 장에서 고유치(eigenvalue) 문제를 세 가지 방법으로 살펴보고 있다.<br>
This chapter looks at the eigenvalue problem through three methods.

* **거듭제곱법 Power Method** ([`200`](./200_Eigenvalues_of_a_Matrix_PowerMethod.ipynb)) : 가장 큰 고유치 **하나** 와 그 고유벡터를 찾는다.<br>finds the **single** dominant eigenvalue and its eigenvector.
* **자코비법 Jacobi Method** ([`240`](./240_Eigenvalue_Jacobi_Method_numpy.ipynb)) : **대칭** 행렬을 대각화하여 **모든** 고유치를 한꺼번에 구한다.<br>diagonalizes a **symmetric** matrix to get **all** eigenvalues at once.
* **QR 알고리듬 QR Algorithm** (이 노트북 / this notebook) : **일반** 행렬(비대칭, 복소수 고유치 포함)의 **모든** 고유치를 구한다.<br>finds **all** eigenvalues of a **general** matrix — non-symmetric, even with complex eigenvalues.

QR 알고리듬은 "일반 행렬의 모든 고유치" 라는 빈자리를 채운다.<br>
The QR algorithm fills the gap of *all eigenvalues of a general matrix*.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as nl

import matshow


## 반복의 아이디어<br>The idea of the iteration

행렬 $A_k$ 를 직교행렬 $Q_k$ 와 상삼각행렬 $R_k$ 의 곱으로 분해(QR 분해)한 뒤, 그 둘을 **반대 순서로** 다시 곱한다.<br>
Factor $A_k$ into an orthogonal $Q_k$ and an upper-triangular $R_k$ (the QR factorization), then multiply them back **in the opposite order**.

$$
A_k = Q_k R_k, \qquad A_{k+1} = R_k Q_k
$$

QR 분해 자체는 [`40_linear_algebra_1`](../40_linear_algebra_1) 의 그람-슈미트 과정에서 다룬다.<br>
The QR factorization itself is covered by the Gram–Schmidt process in `40_linear_algebra_1`.

여기서 핵심은, 새 행렬 $A_{k+1}$ 이 $A_k$ 와 **같은 고유치** 를 가진다는 점이다.<br>
The key point is that the new matrix $A_{k+1}$ has the **same eigenvalues** as $A_k$.

$$
A_{k+1} = R_k Q_k = (Q_k^T Q_k) R_k Q_k = Q_k^T (Q_k R_k) Q_k = Q_k^T A_k Q_k
$$

즉 매 단계는 직교 **상사변환**(similarity transformation) 이다. 자코비법과 마찬가지로 고유치는 보존된다.<br>
So each step is an orthogonal **similarity transformation** — exactly as in the Jacobi method, the eigenvalues are preserved.


## 왜 수렴하는가<br>Why it converges

거듭제곱법은 벡터 하나에 $A$ 를 거듭 곱해 지배적 고유벡터로 수렴시켰다. QR 반복은 이를 **모든 열에 동시에** 적용한 것으로 볼 수 있다 (*부분공간 반복*, subspace iteration).<br>
The power method multiplied $A$ onto one vector repeatedly until it lined up with the dominant eigenvector. QR iteration does this to **all columns at once** (*subspace iteration*).

매 단계 $Q_k$ 로 열들을 다시 정규직교화하기 때문에, 모든 열이 똑같이 지배적 고유벡터로 무너지지 않고 서로 다른 고유방향으로 정렬된다. 그 결과 아래쪽 삼각형의 성분들이 점점 0 이 되고, 행렬은 (준)상삼각행렬로 수렴한다. 이 형태를 **슈어 형식**(Schur form) 이라 한다.<br>
Re-orthonormalizing the columns with $Q_k$ each step keeps them from all collapsing onto the same dominant eigenvector; instead they line up with distinct eigen-directions. The lower-triangular entries die away and the matrix converges to (quasi-)upper-triangular form — the **Schur form** — with the eigenvalues on the diagonal.


In [ ]:
def qr_iteration(matA, n_iter=50, snapshots=None):
    '''
    QR 알고리듬: A = QR 분해 후 A <- RQ 를 반복한다.
    QR algorithm: repeatedly factor A = QR, then form A <- RQ.
    snapshots 리스트를 주면 매 단계 행렬 사본을 모은다 (슬라이더용).
    If a `snapshots` list is given, a copy of A at each step is collected.
    '''
    A = np.array(matA, dtype=float)
    if snapshots is not None:
        snapshots.append(A.copy())

    for _ in range(n_iter):
        Q, R = nl.qr(A)
        A = R @ Q
        if snapshots is not None:
            snapshots.append(A.copy())

    return A


## 사례 1 : 실수 고유치<br>Example 1 : real eigenvalues

고유치가 $\{4, 2, 1\}$ 인 상삼각행렬 $T$ 를 직교 상사변환으로 "섞어" 일반 행렬 $A_0$ 를 만든다. $A_0$ 는 더 이상 삼각행렬이 아니지만 고유치는 그대로다.<br>
Take an upper-triangular $T$ with eigenvalues $\{4, 2, 1\}$ and "scramble" it with an orthogonal similarity into a full matrix $A_0$ — no longer triangular, but with the same eigenvalues.


In [ ]:
# 고정된 직교행렬 (재현 가능) / a fixed orthogonal matrix (reproducible)
Q_scramble, _ = nl.qr(np.array([[1., 2, 0], [0, 1, 3], [2, 0, 1]]))

T = np.array([
    [4., 1, 2],
    [0,  2, 1],
    [0,  0, 1],
])
A0 = Q_scramble @ T @ Q_scramble.T
A0


In [ ]:
snapshots_real = []
A_final = qr_iteration(A0, n_iter=50, snapshots=snapshots_real)
np.set_printoptions(precision=4, suppress=True)
A_final


아래 삼각형 성분이 0 으로 사라지고, 대각선에 고유치가 (크기 큰 것부터) 나타난다.<br>
The lower-triangular entries have vanished and the eigenvalues appear on the diagonal (largest magnitude first).


In [ ]:
print('QR diagonal :', np.sort(np.diag(A_final))[::-1])
print('nl.eigvals  :', np.sort(nl.eigvals(A0).real)[::-1])


### 반복 과정 살펴보기<br>Scrubbing through the iterations

슬라이더를 움직이며 아래쪽 삼각형의 상자(성분)들이 점점 작아져 사라지는 모습을 보자. 모든 프레임이 같은 크기 기준을 쓰므로 단계 사이의 변화를 직접 비교할 수 있다.<br>
Drag the slider and watch the lower-triangular boxes shrink to nothing. Every frame shares one scale, so changes between steps are directly comparable.


In [ ]:
matshow.hinton_step_slider(snapshots_real, description='step')


대각 성분이 고유치로 수렴하는 값을 직접 추적해 볼 수도 있다.<br>
We can also trace the diagonal entries converging to the eigenvalues.


In [ ]:
ax = matshow.element_trace(
    snapshots_real,
    indices=[(0, 0), (1, 1), (2, 2)],
    labels=['a[0][0]', 'a[1][1]', 'a[2][2]'],
)
ax.set_title('diagonal entries converging to eigenvalues 4, 2, 1')
plt.show()


## 수렴 속도와 이동(shift)<br>Convergence rate and shifts

수렴은 기하급수적이며, 그 속도는 이웃한 고유치의 크기비 $\left|\lambda_{i+1}/\lambda_i\right|$ 가 결정한다. 크기가 비슷한 고유치들은 천천히 분리된다.<br>
Convergence is geometric, at a rate set by the ratio of neighbouring eigenvalues $\left|\lambda_{i+1}/\lambda_i\right|$. Eigenvalues of similar magnitude separate slowly.

실제 구현은 이를 가속하기 위해 *이동*(shift) 을 쓴다: $A_k - \mu_k I = Q_k R_k$, $A_{k+1} = R_k Q_k + \mu_k I$. 적절한 $\mu_k$ 는 크기비를 키워 수렴을 크게 앞당긴다. (이 노트북은 이동 없는 기본형만 보인다.)<br>
Practical implementations accelerate this with a *shift*: $A_k - \mu_k I = Q_k R_k$, $A_{k+1} = R_k Q_k + \mu_k I$. A well-chosen $\mu_k$ widens the ratio and speeds convergence dramatically. (This notebook shows only the shift-free basic form.)


## 사례 2 : 복소수 고유치와 2×2 블록<br>Example 2 : complex eigenvalues and 2×2 blocks

**실수** 행렬도 복소수 고유치를 (켤레쌍으로) 가질 수 있다. 그런데 실수 QR 반복은 실수 산술만 쓰므로, 복소수를 실수 대각성분 하나에 올려놓을 수 없다. 대신 켤레쌍 $a \pm bi$ 는 대각선 위의 **$2 \times 2$ 실수 블록** 으로 살아남는다 — 그 블록의 고유치가 바로 그 켤레쌍이다.<br>
A **real** matrix can still have complex eigenvalues, in conjugate pairs. But real QR iteration uses real arithmetic only, so it cannot place a complex number on a single real diagonal entry. Instead a conjugate pair $a \pm bi$ survives as a **$2 \times 2$ real block** on the diagonal — the eigenvalues of that block *are* the pair.

고유치가 $\{5,\ 1+2i,\ 1-2i\}$ 인 행렬로 확인해 보자.<br>
Let's check with a matrix whose eigenvalues are $\{5,\ 1+2i,\ 1-2i\}$.


In [ ]:
S = np.array([
    [5., 1,  1],
    [0,  1, -2],
    [0,  2,  1],
])
A0_complex = Q_scramble @ S @ Q_scramble.T

snapshots_complex = []
A_complex_final = qr_iteration(A0_complex, n_iter=60, snapshots=snapshots_complex)
A_complex_final


행렬은 완전한 상삼각형이 아니라 **준상삼각형**(quasi-upper-triangular) 으로 수렴했다: 오른쪽 아래 $2 \times 2$ 블록의 비대각 성분이 0 이 되지 않고 남아 있다. 이것이 복소수 켤레쌍의 신호다. 대각선만 읽으면 $[5, 1, 1]$ 이라 **오해** 하기 쉽다 — $1, 1$ 은 고유치가 아니라 복소수 고유치의 실수부일 뿐이다.<br>
The matrix converged not to a full triangle but to a **quasi-upper-triangular** form: the off-diagonal of the bottom-right $2 \times 2$ block did *not* go to zero. That is the signature of a complex conjugate pair. Reading the diagonal alone gives $[5, 1, 1]$, which is **misleading** — the $1, 1$ are not eigenvalues but the real parts of the complex pair.


In [ ]:
# 오른쪽 아래 2x2 블록의 고유치가 켤레쌍이다 / the 2x2 block's eigenvalues are the conjugate pair
block_2x2 = A_complex_final[1:, 1:]
print('top-left (real) eigenvalue :', A_complex_final[0, 0])
print('2x2 block eigenvalues      :', nl.eigvals(block_2x2))
print('nl.eigvals(A0_complex)     :', nl.eigvals(A0_complex))


### 실수 슈어 형식 vs 복소수 슈어 형식<br>Real vs complex Schur form

* **복소수 슈어 형식** 은 완전한 상삼각행렬이지만 복소수 산술이 필요하다.<br>
  The **complex Schur form** is fully upper-triangular, but requires complex arithmetic.
* **실수 슈어 형식** 은 실수 산술만으로 얻으며, 복소수 켤레쌍마다 $2 \times 2$ 블록을 남긴다.<br>
  The **real Schur form** stays in real arithmetic, leaving a $2 \times 2$ block per conjugate pair.

교과서와 실제 라이브러리가 보통 실수 슈어 형식을 보이는 까닭은, 실수 입력에 실수 산술을 유지하면서도 $2 \times 2$ 블록이 물리적으로 해석 가능한 형태(예: 진동의 감쇠·진동수)로 남기 때문이다.<br>
Textbooks and libraries usually show the real Schur form because it keeps real inputs in real arithmetic while the $2 \times 2$ blocks remain physically interpretable (e.g. the damping/frequency of an oscillation).


In [ ]:
matshow.hinton_step_slider(snapshots_complex, description='step')


슬라이더를 끝까지 옮겨 보면, 1열 아래 성분들은 사라지지만 오른쪽 아래 $2 \times 2$ 블록은 끝까지 남는 것을 볼 수 있다.<br>
Scrub to the end: the entries below the first column vanish, but the bottom-right $2 \times 2$ block persists all the way through.


## 더 읽을거리 : QR 에서 QZ 로<br>Further reading : from QR to QZ

QR 알고리듬은 1961년 무렵 [John G. F. Francis](https://en.wikipedia.org/wiki/John_G._F._Francis) 와 [Vera N. Kublanovskaya](https://en.wikipedia.org/wiki/Vera_Kublanovskaya) 가 서로 독립적으로 고안했다.<br>
The QR algorithm was devised independently around 1961 by John G. F. Francis and Vera N. Kublanovskaya.

이 표준 고유치 문제 $A\mathbb{x} = \lambda \mathbb{x}$ 를 **일반화 고유치 문제** $A\mathbb{x} = \lambda B\mathbb{x}$ 로 확장한 것이 **QZ 알고리듬** 이다. MATLAB 을 만든 [Cleve Moler](../IN_MEMORIAM_Cleve_Moler.md) 가 G. W. Stewart 와 함께 1973년에 발표했으며, $B = I$ 이면 QR 알고리듬으로 되돌아간다.<br>
Extending this standard eigenvalue problem $A\mathbb{x} = \lambda \mathbb{x}$ to the **generalized eigenvalue problem** $A\mathbb{x} = \lambda B\mathbb{x}$ gives the **QZ algorithm**, published in 1973 by Cleve Moler — the creator of MATLAB — together with G. W. Stewart. When $B = I$ it reduces to the QR algorithm.

> C. B. Moler and G. W. Stewart, *An Algorithm for Generalized Matrix Eigenvalue Problems*, SIAM Journal on Numerical Analysis, **10**(2), 1973.

`scipy.linalg.eig` 가 일반화 고유치 문제를 풀 때 내부적으로 바로 이 QZ 분해를 쓴다.<br>
`scipy.linalg.eig` uses exactly this QZ decomposition internally when solving the generalized problem.


## 정리<br>Summary

* QR 알고리듬은 직교 상사변환 $A_{k+1} = R_k Q_k = Q_k^T A_k Q_k$ 를 반복한다.<br>
  The QR algorithm iterates the orthogonal similarity $A_{k+1} = R_k Q_k = Q_k^T A_k Q_k$.
* 일반 행렬을 (준)상삼각인 **슈어 형식** 으로 수렴시켜 **모든** 고유치를 대각선에서 읽게 한다.<br>
  It drives a general matrix to the (quasi-)upper-triangular **Schur form**, reading **all** eigenvalues off the diagonal.
* 복소수 켤레쌍은 $2 \times 2$ 실수 블록(실수 슈어 형식)으로 남는다.<br>
  Complex conjugate pairs remain as $2 \times 2$ real blocks (the real Schur form).
* 거듭제곱법(고유벡터 하나) → 자코비법(대칭 행렬 대각화) → QR(일반 행렬의 모든 고유치) 로 이어진다.<br>
  Power method (one eigenvector) → Jacobi (diagonalize a symmetric matrix) → QR (all eigenvalues of a general matrix).

### 참고문헌 References

* Wikipedia contributors, [QR algorithm](https://en.wikipedia.org/wiki/QR_algorithm).
* G. H. Golub and C. F. Van Loan, *Matrix Computations*.
* G. Golub and F. Uhlig, *The QR algorithm: 50 years later — its genesis by John Francis and Vera Kublanovskaya and subsequent developments*, IMA Journal of Numerical Analysis, **29**(3), 2009.


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");

